# 02: Radio Broadcast Capture & Crisis Signal Extraction

This notebook demonstrates the radio pipeline — the unique component of CrisisCortex.

Most crisis prediction systems use only satellite data. We add **local radio broadcasts** to detect emerging crises before formal reports exist.

In [ ]:
import sys
sys.path.append('../src')

from pathlib import Path
import tempfile

import numpy as np
import matplotlib.pyplot as plt
import soundfile as sf

from crisiscortex.data.radio import MockRadioCapture, RadioTranscriber

## Part 1: Mock Radio Capture

Without an RTL-SDR dongle, we use `MockRadioCapture` to simulate the hardware layer.

In production, `RadioCapture` connects to a real $20 RTL-SDR USB dongle tuned to local FM stations.

In [ ]:
# Initialize mock capture
mock = MockRadioCapture()

# Show monitoring frequencies for crisis-prone regions
freqs = mock.list_available_frequencies()
print("Crisis Monitoring Frequencies:")
print("=" * 40)
for name, freq in freqs.items():
    print(f"  {name:20s}: {freq/1e6:.1f} MHz")

# Capture a mock broadcast (creates silent audio file for demo)
with tempfile.TemporaryDirectory() as tmpdir:
    output = Path(tmpdir) / "broadcast_95.5MHz.wav"
    captured = mock.capture_to_file(
        frequency_hz=95.5e6,
        duration_seconds=5,
        output_path=output,
    )
    print(f"\nMock capture saved to: {captured}")
    
    # Verify it's a valid audio file
    data, sr = sf.read(captured)
    print(f"Sample rate: {sr} Hz")
    print(f"Duration: {len(data)/sr:.1f} seconds")
    print(f"Samples: {len(data):,}")

## Part 2: Whisper Transcription

Load the Whisper model and transcribe audio. We use `tiny` for speed in this demo; production uses `small` or `base` for better accuracy.

Since we don't have real radio audio yet, we'll test the transcription pipeline on **simulated text** that represents what a real crisis broadcast might contain.

In [ ]:
# Load transcriber (downloads model on first run)
print("Loading Whisper model...")
transcriber = RadioTranscriber("tiny")
print("Model ready.")

## Part 3: Simulated Crisis Broadcast Analysis

This text simulates what Whisper would transcribe from a real FM radio broadcast in the Sahel region during a food crisis.

In [ ]:
# Simulated transcript from a real crisis broadcast
simulated_broadcast = """
Good evening. This is Radio Bamako on 95.5 FM. Today's market report from 
Timbuktu: grain prices have doubled since last month. The millet harvest 
has failed due to locusts and drought. Many families are hungry. The 
hospital in Gao reports an increase in fever cases, possibly cholera. 
Armed groups have been seen moving near the eastern road. Residents are 
advised to stay indoors after dark. International aid organizations 
have not yet arrived in the region. Farmers report their crops are dying 
from lack of rain. The market is empty. People are fleeing to the south.
"""

print("=" * 60)
print("SIMULATED BROADCAST TRANSCRIPT")
print("=" * 60)
print(simulated_broadcast[:300] + "...")
print(f"\nTotal characters: {len(simulated_broadcast):,}")

## Part 4: Extract Crisis Signals

In [ ]:
# Extract crisis signals from the transcript
signals = transcriber.extract_crisis_signals(simulated_broadcast)

print("=" * 60)
print("CRISIS SIGNALS DETECTED")
print("=" * 60)

if not signals:
    print("No crisis signals detected.")
else:
    for crisis_type, keywords in signals.items():
        print(f"\n{crisis_type.upper().replace('_', ' ')}")
        print("-" * 40)
        for kw in keywords:
            print(f"  • '{kw}'")
        print(f"  → {len(keywords)} keyword(s) matched")

# Calculate severity score
total_keywords = sum(len(v) for v in signals.values())
severity = min(total_keywords / 5.0, 1.0)

print(f"\n{'=' * 60}")
print(f"SEVERITY SCORE: {severity:.2f} / 1.00")
print(f"ALERT TRIGGERED: {'YES' if severity > 0.3 else 'NO'}")
print(f"{'=' * 60}")

## Part 5: Visualize Crisis Signals

In [ ]:
# Bar chart of signal counts by crisis type
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Chart 1: Keyword counts
if signals:
    crisis_types = list(signals.keys())
    counts = [len(signals[ct]) for ct in crisis_types]
    display_names = [ct.replace('_', ' ').title() for ct in crisis_types]
    colors = ['#e74c3c', '#f39c12', '#9b59b6']
    
    bars = axes[0].bar(display_names, counts, color=colors[:len(counts)], 
                       edgecolor='black', linewidth=1.5)
    axes[0].set_ylabel('Keyword Matches', fontsize=12)
    axes[0].set_title('Crisis Signals Detected in Broadcast', fontsize=14, weight='bold')
    axes[0].set_ylim(0, max(counts) + 1)
    
    for bar, count in zip(bars, counts):
        height = bar.get_height()
        axes[0].text(bar.get_x() + bar.get_width()/2., height + 0.1,
                    f'{count}', ha='center', va='bottom', fontsize=14, weight='bold')
else:
    axes[0].text(0.5, 0.5, 'No Signals Detected', ha='center', va='center', 
                 fontsize=14, transform=axes[0].transAxes)
    axes[0].set_title('Crisis Signals', fontsize=14)

# Chart 2: Severity gauge
gauge_colors = ['#2ecc71', '#f1c40f', '#e67e22', '#e74c3c']
gauge_labels = ['Low\n(0-0.3)', 'Medium\n(0.3-0.5)', 'High\n(0.5-0.8)', 'Critical\n(0.8-1.0)']

for i, (color, label) in enumerate(zip(gauge_colors, gauge_labels)):
    axes[1].barh(0, 0.25, left=i*0.25, color=color, edgecolor='black', height=0.5)
    axes[1].text(i*0.25 + 0.125, -0.4, label, ha='center', va='top', fontsize=9)

# Severity indicator
axes[1].scatter([severity], [0], s=300, c='black', zorder=5, marker='v')
axes[1].text(severity, 0.35, f'{severity:.2f}', ha='center', va='bottom', 
             fontsize=16, weight='bold')

axes[1].set_xlim(0, 1)
axes[1].set_ylim(-0.8, 0.6)
axes[1].set_title('Severity Assessment', fontsize=14, weight='bold')
axes[1].axis('off')

plt.tight_layout()
plt.show()

## Part 6: Full Analysis Pipeline

In [ ]:
# Simulate the full pipeline output
full_analysis = {
    "source": "Radio Bamako, 95.5 MHz",
    "region": "sahel_mali",
    "timestamp": "2024-03-15T18:30:00Z",
    "language": "fr",  # French is common in Mali
    "transcript_preview": simulated_broadcast[:200] + "...",
    "crisis_signals": signals,
    "severity_score": round(severity, 2),
    "alert_triggered": severity > 0.3,
    "recommended_action": "Escalate to FEWS NET and WHO if severity > 0.5",
}

print("=" * 60)
print("CRISISCORTEX ANALYSIS REPORT")
print("=" * 60)
for key, value in full_analysis.items():
    if key == "crisis_signals":
        print(f"\n{key}:")
        for ct, kws in value.items():
            print(f"  {ct}: {kws}")
    else:
        print(f"{key:20s}: {value}")

print(f"\n{'=' * 60}")
print("NEXT STEPS:")
print("  1. Cross-reference with satellite NDVI for same region")
print("  2. Check night-lights for economic activity changes")
print("  3. If 2+ modalities agree → HIGH CONFIDENCE ALERT")
print(f"{'=' * 60}")

## Part 7: Compare Multiple Broadcasts

Simulate how the system tracks signals over time — increasing severity indicates escalating crisis.

In [ ]:
# Simulate broadcasts over 4 weeks
weekly_broadcasts = [
    "Market prices are stable. Weather is normal. Harvest expected soon.",
    "Some concern about late rains. Prices slightly higher than last month.",
    "Market empty. Prices doubled. Farmers report crops dying from drought.",
    "Famine conditions reported. Hospital full of fever cases. People fleeing.",
]

weekly_scores = []
for i, broadcast in enumerate(weekly_broadcasts):
    sigs = transcriber.extract_crisis_signals(broadcast)
    score = min(sum(len(v) for v in sigs.values()) / 5.0, 1.0)
    weekly_scores.append(score)
    print(f"Week {i+1}: Severity = {score:.2f} | Signals: {list(sigs.keys())}")

# Plot trend
fig, ax = plt.subplots(figsize=(10, 5))
weeks = range(1, len(weekly_scores) + 1)

ax.plot(weeks, weekly_scores, 'o-', linewidth=3, markersize=10, color='#e74c3c')
ax.axhline(y=0.3, color='orange', linestyle='--', label='Alert Threshold')
ax.axhline(y=0.6, color='red', linestyle='--', label='Critical Threshold')

ax.fill_between(weeks, 0, weekly_scores, alpha=0.3, color='#e74c3c')
ax.set_xlabel('Week', fontsize=12)
ax.set_ylabel('Severity Score', fontsize=12)
ax.set_title('Crisis Escalation Over Time (Simulated)', fontsize=14, weight='bold')
ax.set_ylim(0, 1)
ax.set_xticks(weeks)
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nTrend: Severity increased from {weekly_scores[0]:.2f} to {weekly_scores[-1]:.2f}")
print(f"Lead time: {(next((i for i, s in enumerate(weekly_scores) if s > 0.3), -1) + 1)} weeks before critical threshold")

## Summary

This notebook demonstrated:

1. **Mock radio capture** — simulates RTL-SDR hardware
2. **Whisper transcription** — converts audio to text
3. **Crisis signal extraction** — finds food/disease/conflict keywords
4. **Severity scoring** — quantifies crisis level
5. **Trend analysis** — tracks escalation over time

In production, this pipeline runs on a Raspberry Pi with a real RTL-SDR dongle, processing local FM broadcasts in real-time. The text output feeds into the cross-modal fusion model alongside satellite and night-lights data.